# Data Collection and Preparation

## Environmental Degradation and Well-being Analysis

### Objectives:
1. Collect data from multiple authoritative sources (EPA, World Bank, UN, etc.)
2. Document data provenance and characteristics
3. Perform initial data quality assessment
4. Clean and preprocess datasets
5. Integrate data from multiple sources
6. Create master analytical datasets

### Data Sources:
- World Bank Open Data (CO2 emissions, energy use, etc.)
- Environmental Protection Agency (EPA)
- United Nations Environment Programme (UNEP)
- International Energy Agency (IEA)
- Our World in Data
- Additional sources as identified

### Output:
- Cleaned datasets saved in `../data/processed/`
- Data dictionary documenting all variables
- Data quality report

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries imported successfully")
print(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Libraries imported successfully
Analysis Date: 2026-01-14 21:02:52


## 1. Data Collection

### 1.1 Download and Load Raw Data

In [2]:
!pip install wbdata

Defaulting to user installation because normal site-packages is not writeable

   ------------------------ --------------- 3/5 [dateparser]
   ------------------------ --------------- 3/5 [dateparser]
   ------------------------ --------------- 3/5 [dateparser]
   ------------------------ --------------- 3/5 [dateparser]
   ------------------------ --------------- 3/5 [dateparser]
   -------------------------------- ------- 4/5 [wbdata]
   ---------------------------------------- 5/5 [wbdata]



  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [ ]:
# Download data from identified sources
# - World Bank API: https://api.worldbank.org/v2/
# - Kaggle datasets
# - Government data portals

import wbdata
import pandas as pd
import datetime
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("DATA COLLECTION MODULE")
print("=" * 80)

# Create directories if they don't exist
data_raw_dir = Path('../data/raw')
data_raw_dir.mkdir(parents=True, exist_ok=True)

# ============================================================================
# 1. WORLD BANK API DATA COLLECTION
# ============================================================================
print("\n[1/3] Collecting data from World Bank API...")
print("Note: This may take several minutes...")

# Define World Bank indicators (using verified indicator codes)
wb_indicators = {
    # Environmental Indicators
    'EN.ATM.CO2E.PC': 'CO2_emissions_per_capita',
    'EG.USE.PCAP.KG.OE': 'energy_use_per_capita',
    'AG.LND.FRST.ZS': 'forest_area_percent',
    'EN.ATM.PM25.MC.M3': 'pm25_air_pollution',
    'AG.LND.AGRI.ZS': 'agricultural_land_percent',
    
    # Economic Indicators
    'NY.GDP.PCAP.CD': 'gdp_per_capita',
    'NY.GDP.MKTP.CD': 'gdp_total',
    'NY.GDP.MKTP.KD.ZG': 'gdp_growth',
    
    # Social/Well-being Indicators
    'SP.POP.TOTL': 'population_total',
    'SP.URB.TOTL.IN.ZS': 'urban_population_percent',
    'SP.DYN.LE00.IN': 'life_expectancy',
    'SH.DYN.MORT': 'mortality_rate',
    'SE.XPD.TOTL.GD.ZS': 'education_expenditure_pct_gdp',
    'SH.XPD.CHEX.GD.ZS': 'health_expenditure_pct_gdp',
    'SI.POV.GINI': 'gini_index',
}

# Collect data from World Bank
wb_dataframes = {}
successful_indicators = []
failed_indicators = []

for indicator_code, indicator_name in wb_indicators.items():
    try:
        print(f"  [{len(successful_indicators)+1}/{len(wb_indicators)}] Downloading: {indicator_name}...", end=" ")
        # Download data (wbdata automatically gets all available years and countries)
        data = wbdata.get_dataframe({indicator_code: indicator_name})
        
        if not data.empty:
            wb_dataframes[indicator_name] = data
            successful_indicators.append(indicator_name)
            print(f"OK ({len(data):,} records)")
        else:
            failed_indicators.append(indicator_name)
            print("WARN: No data returned")
    except Exception as e:
        failed_indicators.append(indicator_name)
        error_msg = str(e)[:60]
        print(f"ERROR: {error_msg}")

print(f"\n  Successfully downloaded: {len(successful_indicators)}/{len(wb_indicators)} indicators")
if failed_indicators:
    print(f"  Failed indicators: {', '.join(failed_indicators)}")

# Combine all World Bank data into a single dataframe
if wb_dataframes:
    print("\n  Combining World Bank data...")
    wb_combined = pd.concat(wb_dataframes.values(), axis=1)
    wb_combined = wb_combined.reset_index()
    
    # Clean up the dataframe
    if 'country' in wb_combined.columns:
        wb_combined.rename(columns={'country': 'country_name'}, inplace=True)
    if 'date' in wb_combined.columns:
        wb_combined.rename(columns={'date': 'year'}, inplace=True)
    
    # Save to CSV
    wb_output_path = data_raw_dir / 'worldbank_data.csv'
    wb_combined.to_csv(wb_output_path, index=False)
    print(f"  Saved World Bank data: {wb_output_path}")
    print(f"    Shape: {wb_combined.shape[0]:,} rows x {wb_combined.shape[1]} columns")
    
    if 'year' in wb_combined.columns:
        years = wb_combined['year'].dropna()
        if len(years) > 0:
            print(f"    Years: {years.min()} to {years.max()}")
    
    if 'country_name' in wb_combined.columns:
        print(f"    Countries: {wb_combined['country_name'].nunique()}")
    
    print(f"    Columns: {', '.join(list(wb_combined.columns)[:5])}...")
else:
    print("  WARNING: No World Bank data was successfully downloaded")

In [9]:

# ============================================================================
# 2. KAGGLE DATASETS
# ============================================================================
print("\n[2/3] Collecting data from Kaggle...")

# Check for manually downloaded Kaggle files
kaggle_files_expected = [
    'world_happiness.csv',
    'climate_change.csv',
    'environmental_indicators.csv'
]

environmental_indicators = pd.read_csv('https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Data/regression_sprint/enviro_indicators.csv', index_col=0)
wb_output_path = data_raw_dir / 'enviro_indicators.csv'
environmental_indicators.to_csv(wb_output_path, index=False)



[2/3] Collecting data from Kaggle...


In [5]:
# ============================================================================
# 3. GOVERNMENT DATA PORTALS
# ============================================================================
print("\n[3/3] Government Data Portals...")
print("  Note: Government data typically requires manual download")
print("  Sources:")
print("    - EPA: epa.gov/enviro")
print("    - UNEP: unep.org/resources")
print("    - OECD: stats.oecd.org")
print("    - WHO: who.int/data/gho")
print(f"\n  Place CSV files in: {data_raw_dir}")

# Check for government data files
gov_files_expected = [
    'epa_emissions.csv',
    'unep_environmental.csv',
    'oecd_indicators.csv'
]

DATA COLLECTION MODULE

[1/3] Collecting data from World Bank API...
Note: This may take several minutes...
  [1/15] Downloading: CO2_emissions_per_capita... ERROR: Got error 175 (Invalid format): The indicator was not found.
  [1/15] Downloading: energy_use_per_capita... OK (17,290 records)
  [2/15] Downloading: forest_area_percent... OK (17,290 records)
  [3/15] Downloading: pm25_air_pollution... OK (17,290 records)
  [4/15] Downloading: agricultural_land_percent... OK (17,290 records)
  [5/15] Downloading: gdp_per_capita... OK (17,290 records)
  [6/15] Downloading: gdp_total... OK (17,290 records)
  [7/15] Downloading: gdp_growth... OK (17,290 records)
  [8/15] Downloading: population_total... OK (17,290 records)
  [9/15] Downloading: urban_population_percent... OK (17,290 records)
  [10/15] Downloading: life_expectancy... OK (17,290 records)
  [11/15] Downloading: mortality_rate... OK (17,290 records)
  [12/15] Downloading: education_expenditure_pct_gdp... OK (17,290 records)
  [13

KeyboardInterrupt: 

## 2. Data Quality Assessment

### 2.1 Check for Missing Values
### 2.2 Identify Outliers
### 2.3 Assess Data Completeness

In [ ]:
# TODO: Implement data quality checks
# - Check for null values
# - Assess data types
# - Identify outliers using statistical methods
# - Evaluate temporal and geographic coverage

print("Data quality assessment in progress...")

## 3. Data Cleaning

### 3.1 Handle Missing Values
### 3.2 Address Outliers
### 3.3 Standardize Formats

In [ ]:
# TODO: Implement data cleaning procedures
# - Decide on strategies for missing values (imputation, deletion)
# - Handle outliers appropriately
# - Standardize date formats, geographic identifiers, units
# - Remove duplicates

print("Data cleaning in progress...")

## 4. Data Integration

### 4.1 Merge Datasets
### 4.2 Create Master Dataset

In [ ]:
# TODO: Integrate data from multiple sources
# - Merge on common keys (country, year, region)
# - Resolve conflicts in overlapping data
# - Create comprehensive master dataset

print("Data integration in progress...")

## 5. Feature Engineering

### 5.1 Create Derived Variables
### 5.2 Calculate Per Capita Metrics

In [ ]:
# TODO: Create new features
# - Per capita emissions
# - Growth rates
# - Efficiency ratios
# - Composite indices

print("Feature engineering in progress...")

## 6. Save Processed Data

In [ ]:
# TODO: Save cleaned and processed datasets
# df_master.to_csv('../data/processed/master_dataset.csv', index=False)
# Save data dictionary

print("Data saved successfully")

## Summary

This notebook completed the following:
- [ ] Data collection from multiple sources
- [ ] Data quality assessment
- [ ] Data cleaning and preprocessing
- [ ] Data integration
- [ ] Feature engineering
- [ ] Creation of master analytical dataset

**Next Steps:** Proceed to Notebook 2 for Exploratory Data Analysis